In [8]:
from dotenv import load_dotenv
import os

In [9]:
#sample input

In [10]:
patient_symptons = ['Fever' , 'Headache' , 'Soreness']

In [11]:
patient_condition = {
    "Age": 67,
    "Mobility Issues": "Requires cane for walking; limited stamina when standing for extended periods",
    "Known Allergies": ["Penicillin", "Latex"],
    "Chronic Conditions": ["Type 2 Diabetes", "Mild Hypertension"],
    "Recent Surgeries": "Hip replacement (8 months ago)",
    "Immunization Status": "Up-to-date, including flu and COVID-19 boosters",
    "Other Notes": "Reports occasional dizziness and sensitivity to cold weather"
}
patient_condition

{'Age': 67,
 'Mobility Issues': 'Requires cane for walking; limited stamina when standing for extended periods',
 'Known Allergies': ['Penicillin', 'Latex'],
 'Chronic Conditions': ['Type 2 Diabetes', 'Mild Hypertension'],
 'Recent Surgeries': 'Hip replacement (8 months ago)',
 'Immunization Status': 'Up-to-date, including flu and COVID-19 boosters',
 'Other Notes': 'Reports occasional dizziness and sensitivity to cold weather'}

In [12]:
geographic_location = {
    "Country": "United States",
    "State": "Kentucky",
    "City": "Highland Heights",
    "Zip Code": "41076",
    "Coordinates": {
        "Latitude": 39.1311,
        "Longitude": -84.5083
    },
    "Preferred Mode of Care": "In person"
}

In [13]:
#The first thing we would calculate is something like nearest health care centers or hospitals based on location

In [15]:
from geopy.distance import geodesic
import pandas as pd

In [16]:
# Sample list of hospitals with coordinates (for demo purposes)
hospital_data = [
    {"Name": "University of Cincinnati Medical Center", "Latitude": 39.1412, "Longitude": -84.5059},
    {"Name": "Cincinnati VA Medical Center", "Latitude": 39.1398, "Longitude": -84.5039},
    {"Name": "Christ Hospital", "Latitude": 39.1231, "Longitude": -84.5070},
    {"Name": "Cleveland Clinic", "Latitude": 41.5033, "Longitude": -81.6200},
    {"Name": "Mount Carmel East", "Latitude": 39.9828, "Longitude": -82.8291}
]
#make a dataframe of this, in reality this will I think be one dataframe per state, consisteing of all health care and hospital services in that area
df_hospitals = pd.DataFrame(hospital_data)


In [17]:
print("df_hospitals can also include things like what services are offered, which department and nuimber of employees; this is bascially a very big table")

df_hospitals can also include things like what services are offered, which department and nuimber of employees; this is bascially a very big table


In [18]:
#One example of  us calculating distance from patient to each hospital
latitude = geographic_location["Coordinates"]["Latitude"]
longitude = geographic_location["Coordinates"]["Longitude"]
patient_coords = (latitude, longitude)
patient_coords

(39.1311, -84.5083)

In [19]:
#I can do many things in order to calculate the actual distance based on latitide and longitide, maybe subtract both and keep the one with the lowest difference
# KNN? Overkill
# I think there might be libraries that do that properly
#geodesic offers a miles bersion that extract miles from distance, which is nice. I a,m going to do that

In [20]:
distances = []
# Calculate distance for each hospital
for index, row in df_hospitals.iterrows():
    hospital_coords = (row["Latitude"], row["Longitude"])
    distance = geodesic(patient_coords, hospital_coords).miles
    distances.append(distance)
distances

[0.7085628278211792,
 0.6450317424197475,
 0.5562695371259393,
 223.71261059390406,
 107.20597664086802]

In [21]:
#The lowest distance is our hospital

In [22]:
df_hospitals["Distance_miles"] = distances
df_hospitals

,Name,Latitude,Longitude,Distance_miles
0,University of Cincinnati Medical Center,39.1412,-84.5059,0.708563
1,Cincinnati VA Medical Center,39.1398,-84.5039,0.645032
2,Christ Hospital,39.1231,-84.5070,0.556270
3,Cleveland Clinic,41.5033,-81.6200,223.712611
4,Mount Carmel East,39.9828,-82.8291,107.205977


In [23]:
closest_index = df_hospitals["Distance_miles"].idxmin() #find the id of the minimum
closest_hospital = df_hospitals.loc[closest_index, "Name"] #look up the name of the minimum
print("The closest hospital is", closest_hospital)
#Actually, gemini is smart enough for now to choose a hoispital for us based on the values as well.

The closest hospital is Christ Hospital


In [24]:
#I think I can use Gemini to build a basic pipeline for me now

In [25]:
#For now I will just use the base model but I can tinker with it later and use some RAG for the customized treatment plan.
#As long as I dont get the treatement plan, I wont be able to fine tune the model.
#Plus base gemini is jsut as fine, can do most of the stuff

In [27]:
#my second version will make a AI agent class but this notebook will only work on the base base base version
from google import genai

In [28]:
load_dotenv()
client = genai.Client(api_key= os.getenv('Gemini_Api_Key'))
client

In [29]:
#Sample response
response = client.models.generate_content(
    model="gemini-2.0-flash", contents="Explain how AI works in a few words"
)
print(response.text)

AI learns patterns from data to make predictions or decisions.



In [30]:
#Now the first step is prompt engineerning. That solves most of our problems

In [31]:
prompt = f'''
You are a healthcare AI assistant tasked with generating a customized treatment plan for a patient. Given the following inputs:
1. List of patient symptoms which is {patient_symptons}
2. Patient's physical condition which is {patient_condition}
3. Geographic location which is {geographic_location}
Also the list of hospitals are listed below: {df_hospitals}
Generate a detailed treatment plan that includes the following:

1.Suggested medical actions: Specify appropriate steps such as scheduling a primary care visit, recommending lab tests or imaging, referrals to specialists, or urgent care if needed.
2.Location-specific considerations: Based on the patient’s location, recommend nearby clinics, hospitals, or telehealth options within a reasonable distance.
3.Justifications: For every suggested action, include a clear, medically-sound justification based on the patient’s symptoms and conditions.

Format your output in three clearly labeled sections:
1. Medical Actions
2. Location-Specific Options
3. Justifications

Be concise, medically accurate, and context-aware. Avoid unnecessary medical jargon.

'''
prompt

"\nYou are a healthcare AI assistant tasked with generating a customized treatment plan for a patient. Given the following inputs:\n1. List of patient symptoms which is ['Fever', 'Headache', 'Soreness']\n2. Patient's physical condition which is {'Age': 67, 'Mobility Issues': 'Requires cane for walking; limited stamina when standing for extended periods', 'Known Allergies': ['Penicillin', 'Latex'], 'Chronic Conditions': ['Type 2 Diabetes', 'Mild Hypertension'], 'Recent Surgeries': 'Hip replacement (8 months ago)', 'Immunization Status': 'Up-to-date, including flu and COVID-19 boosters', 'Other Notes': 'Reports occasional dizziness and sensitivity to cold weather'}\n3. Geographic location which is {'Country': 'United States', 'State': 'Kentucky', 'City': 'Highland Heights', 'Zip Code': '41076', 'Coordinates': {'Latitude': 39.1311, 'Longitude': -84.5083}, 'Preferred Mode of Care': 'In person'}\nAlso the list of hospitals are listed below:                                       Name  Latitu

In [32]:
#Sample response
response = client.models.generate_content(
    model="gemini-2.0-flash", contents= prompt
)
response.text

"Here's a suggested treatment plan based on the provided information:\n\n**1. Medical Actions**\n\n*   **Schedule a Primary Care Visit:** Schedule an in-person appointment with the patient's primary care physician as soon as possible, ideally within 24-48 hours.\n*   **Symptomatic Treatment:** Recommend over-the-counter pain relievers (acetaminophen or ibuprofen) for fever, headache, and soreness, taking into consideration the patient's Type 2 Diabetes and Mild Hypertension. Advise on appropriate dosages and contraindications.\n*   **Check Blood Pressure and Blood Sugar:** During the primary care visit, ensure the patient's blood pressure and blood sugar levels are checked to rule out any exacerbation of their chronic conditions due to the current illness.\n*   **Influenza and COVID-19 Testing:** Administer influenza and COVID-19 tests to determine if the patient has contracted either virus.\n*   **Evaluate Dizziness:** Further evaluate the patient's reported dizziness. This might invo

In [33]:
# Display the output in a clean, formatted manner
print("\n===== Customized Treatment Plan =====\n")

# Split sections if the model followed the prompt structure
sections = response.text.strip().split('\n\n')

for section in sections:
    if section.strip():
        print(section)
        print("-" * 60)  # Divider between sections


===== Customized Treatment Plan =====

Here's a suggested treatment plan based on the provided information:
------------------------------------------------------------
**1. Medical Actions**
------------------------------------------------------------
*   **Schedule a Primary Care Visit:** Schedule an in-person appointment with the patient's primary care physician as soon as possible, ideally within 24-48 hours.
*   **Symptomatic Treatment:** Recommend over-the-counter pain relievers (acetaminophen or ibuprofen) for fever, headache, and soreness, taking into consideration the patient's Type 2 Diabetes and Mild Hypertension. Advise on appropriate dosages and contraindications.
*   **Check Blood Pressure and Blood Sugar:** During the primary care visit, ensure the patient's blood pressure and blood sugar levels are checked to rule out any exacerbation of their chronic conditions due to the current illness.
*   **Influenza and COVID-19 Testing:** Administer influenza and COVID-19 tests 

In [34]:
#Gemini-output looks good enough to me.
#lets work with Langchain on next notebook to create multi agentic models to see if it performs any better.